LABORATÓRIO — ACO: OTIMIZAÇÃO POR COLÔNIA DE FORMIGAS

Objetivo: Implementar uma versão simples do **Ant Colony Optimization (ACO)** para encontrar uma boa rota em uma rede de nós.

O algoritmo utiliza três ideias principais:

1. As formigas exploram diferentes caminhos.
2. Caminhos melhores recebem mais feromônio.
3. O feromônio influencia as próximas escolhas.

In [11]:
import numpy as np
import random
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CUSTOS = np.array([
    [0, 2, 4, np.inf, np.inf, np.inf],
    [2, 0, 1, 5, np.inf, np.inf],
    [4, 1, 0, 2, 3, np.inf],
    [np.inf, 5, 2, 0, 1, 4],
    [np.inf, np.inf, 3, 1, 0, 2],
    [np.inf, np.inf, np.inf, 4, 2, 0]
])

ORIGEM = 0
DESTINO = 5

NUM_FORMIGAS = 20
NUM_ITERACOES = 50

ALPHA = 1.0
BETA = 2.0

TAXA_EVAPORACAO = 0.5
Q = 100

random.seed(42)
np.random.seed(42)

feromonio = np.ones_like(CUSTOS, dtype=float)
feromonio[CUSTOS == np.inf] = 0

def obter_vizinhos(no):
    vizinhos = []
    for proximo in range(len(CUSTOS)):
        if proximo != no and CUSTOS[no][proximo] != np.inf:
            vizinhos.append(proximo)
    return vizinhos

def escolher_proximo(no_atual, visitados):
    vizinhos = obter_vizinhos(no_atual)
    candidatos = [no for no in vizinhos if no not in visitados]
    if not candidatos:
        return None
    atratividades = []
    for proximo in candidatos:
        fer = feromonio[no_atual][proximo]
        custo = CUSTOS[no_atual][proximo]
        atratividade = (fer ** ALPHA) * ((1 / custo) ** BETA)
        atratividades.append(atratividade)
    soma = sum(atratividades)
    probabilidades = [valor / soma for valor in atratividades]
    return random.choices(candidatos, weights=probabilidades, k=1)[0]

def construir_rota():
    rota = [ORIGEM]
    atual = ORIGEM
    while atual != DESTINO:
        proximo = escolher_proximo(atual, rota)
        if proximo is None:
            return None
        rota.append(proximo)
        atual = proximo
    return rota

def calcular_custo(rota):
    total = 0
    for i in range(len(rota) - 1):
        total += CUSTOS[rota[i]][rota[i+1]]
    return total

def depositar_feromonio(rota, custo):
    deposito = Q / custo
    for i in range(len(rota) - 1):
        origem = rota[i]
        destino = rota[i + 1]
        feromonio[origem][destino] += deposito

def evaporar_feromonio():
    global feromonio
    feromonio *= (1 - TAXA_EVAPORACAO)
    feromonio[CUSTOS == np.inf] = 0

melhor_rota = None
melhor_custo = float("inf")
historico = []

for iteracao in range(NUM_ITERACOES):
    rotas = []
    for _ in range(NUM_FORMIGAS):
        rota = construir_rota()
        if rota is not None:
            custo = calcular_custo(rota)
            rotas.append((rota, custo))
            if custo < melhor_custo:
                melhor_custo = custo
                melhor_rota = rota.copy()
    evaporar_feromonio()
    for rota, custo in rotas:
        depositar_feromonio(rota, custo)
    historico.append(melhor_custo)

print("Melhor rota encontrada:", melhor_rota)
print("Melhor custo:", melhor_custo)
print("Historico (a cada 5 iteracoes):", historico[::5])
print("\nMatriz final de feromonio:")
print(np.round(feromonio, 2))

plt.figure(figsize=(10, 5))
plt.plot(historico)
plt.xlabel("Iteração")
plt.ylabel("Melhor custo")
plt.title("Convergência do ACO")
plt.grid()
plt.savefig("convergencia.png", dpi=120)
plt.close()

plt.figure(figsize=(7, 6))
plt.imshow(feromonio, cmap="hot")
plt.colorbar(label="Quantidade de feromônio")
plt.xlabel("Nó de destino")
plt.ylabel("Nó de origem")
plt.title("Memória Coletiva da Colônia")
plt.savefig("feromonio.png", dpi=120)
plt.close()

Melhor rota encontrada: [0, 1, 2, 3, 4, 5]
Melhor custo: 8.0
Historico (a cada 5 iteracoes): [np.float64(8.0), np.float64(8.0), np.float64(8.0), np.float64(8.0), np.float64(8.0), np.float64(8.0), np.float64(8.0), np.float64(8.0), np.float64(8.0), np.float64(8.0)]

Matriz final de feromonio:
[[  0. 500.   0.   0.   0.   0.]
 [  0.   0. 500.   0.   0.   0.]
 [  0.   0.   0. 500.   0.   0.]
 [  0.   0.   0.   0. 500.   0.]
 [  0.   0.   0.   0.   0. 500.]
 [  0.   0.   0.   0.   0.   0.]]


**Resultados da execução**

**Melhor rota encontrada:** [0, 1, 2, 3, 4, 5]

Melhor custo: **negrito** 8.0

**Convergência:** a colônia já encontra a rota ótima (custo 8) logo nas primeiras iterações e o valor se mantém estável dali em diante — a curva fica praticamente reta.

**Matriz final de feromônio:** o feromônio se concentra quase inteiramente nas arestas da rota ótima (0→1, 1→2, 2→3, 3→4, 4→5), todas com valor ~500, enquanto as demais conexões tendem a zero por causa da evaporação.